In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
import numpy as np

In [21]:
!ls /content/drive/MyDrive

'11_Maths_Activity(1-6).pdf'
 20200820_170823.jpg
 20200820_192335.jpg
'2[1].jpg'
'2403104837549945929011 (1).pdf'
'actual shit.gsheet'
'Adobe Scan 7 Nov 2022 (1).pdf'
 BACKLOGS.gdoc
'Backup Dell'
'Bitsat Practice Papers GKP WWW.EXAMSAKHA.IN.pdf'
 BITS.gdoc
 Branch.gsheet
'Card (1).pdf'
 CG.gsheet
 Chemistry.png
 Classroom
'Colab Notebooks'
 COLLEGE.gsheet
'Copy of 1410_2907 Campus Xplore! - Freshers of 2024.pdf'
'CP NOTES.gdoc'
 CS50_DS_CODES.gdoc
'CS50x 2025 Final Project.gslides'
'DSA STRIVER YT PLAYLIST.gsheet'
'Empty Periodic Table .gsheet'
'English Assignment -Artical.docx'
'english poster.pdf'
 Expenses.gsheet
'Getting started.pdf'
 Habits.gsheet
 Hackenza_MUPlovers
'Hackenza PS.gdoc'
 H.gsheet
'hindi project 2.pptx'
'hindi project.pptx'
'History and Civics worksheets (1).pptx'
'History and Civics worksheets (2).pptx'
'History and Civics worksheets.pptx'
'History project.pptx'
'Important 11 & 12 Chapters.gdoc'
 Items.gsheet
'IX and XI 18th to 23rd May 20(1) (1).pdf'
'IX and XI 1

In [22]:
import torch
print("GPU:", torch.cuda.is_available())

GPU: True


In [23]:
!pip install -q transformers==4.38.2

In [24]:
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-large")

model = WavLMModel.from_pretrained(
    "microsoft/wavlm-large",
    output_hidden_states=True
).to(device)

model.eval()

num_layers = model.config.num_hidden_layers + 1

# FIX: move to same device
layer_weights = nn.Parameter(
    torch.ones(num_layers, device=device) / num_layers
)

print("WavLM-Large loaded ✅")
print("Hidden layers:", num_layers)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at microsoft/wavlm-large were not used when initializing WavLMModel: ['encoder.pos_conv_embed.conv.weight_g', 'encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing WavLMModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing WavLMModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of WavLMModel were not initialized from the model checkpoint a

WavLM-Large loaded ✅
Hidden layers: 25


In [25]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/Hackenza_MUPlovers"

DATA_PATH = os.path.join(DRIVE_ROOT, "data")
PROCESSED_AUDIO_PATH = os.path.join(DATA_PATH, "processed")
CHUNK_INDEX_PATH = os.path.join(DATA_PATH, "chunk_index.csv")

EMBED_SAVE_PATH = os.path.join(DRIVE_ROOT, "cache", "embeddings")
os.makedirs(EMBED_SAVE_PATH, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Processed audio path:", PROCESSED_AUDIO_PATH)
print("Chunk index path:", CHUNK_INDEX_PATH)
print("Embedding save path:", EMBED_SAVE_PATH)

Drive root: /content/drive/MyDrive/Hackenza_MUPlovers
Processed audio path: /content/drive/MyDrive/Hackenza_MUPlovers/data/processed
Chunk index path: /content/drive/MyDrive/Hackenza_MUPlovers/data/chunk_index.csv
Embedding save path: /content/drive/MyDrive/Hackenza_MUPlovers/cache/embeddings


In [26]:
!ls /content/drive/MyDrive/Hackenza_MUPlovers/data/processed

288.wav  341.wav  379.wav  414.wav  466.wav  501.wav  552.wav  601.wav	655.wav
294.wav  342.wav  380.wav  416.wav  468.wav  502.wav  560.wav  605.wav	656.wav
298.wav  343.wav  382.wav  419.wav  473.wav  504.wav  565.wav  606.wav	657.wav
299.wav  344.wav  383.wav  420.wav  477.wav  510.wav  566.wav  607.wav	663.wav
309.wav  345.wav  386.wav  430.wav  478.wav  514.wav  568.wav  609.wav	668.wav
311.wav  351.wav  387.wav  431.wav  479.wav  517.wav  569.wav  617.wav	680.wav
316.wav  352.wav  389.wav  433.wav  480.wav  520.wav  571.wav  618.wav	682.wav
317.wav  355.wav  390.wav  435.wav  481.wav  521.wav  573.wav  621.wav	689.wav
319.wav  357.wav  393.wav  440.wav  484.wav  522.wav  574.wav  625.wav	696.wav
322.wav  360.wav  394.wav  441.wav  485.wav  525.wav  576.wav  626.wav	697.wav
326.wav  362.wav  396.wav  446.wav  487.wav  529.wav  578.wav  628.wav	703.wav
328.wav  363.wav  398.wav  451.wav  490.wav  530.wav  579.wav  632.wav	716.wav
331.wav  367.wav  402.wav  452.wav  492.wav  532.wav

In [27]:
import pandas as pd

chunk_df = pd.read_csv(CHUNK_INDEX_PATH)

print(chunk_df.head())
print("\nColumns:", chunk_df.columns)

   file_id  chunk_id  start_sec  end_sec  start_sample  end_sample  \
0      288         0        0.0      3.0             0       48000   
1      288         1        1.5      4.5         24000       72000   
2      288         2        3.0      6.0         48000       96000   
3      288         3        4.5      7.5         72000      120000   
4      288         4        6.0      9.0         96000      144000   

   chunk_len_samples  is_padded  file_total_samples  
0              48000          0              556800  
1              48000          0              556800  
2              48000          0              556800  
3              48000          0              556800  
4              48000          0              556800  

Columns: Index(['file_id', 'chunk_id', 'start_sec', 'end_sec', 'start_sample',
       'end_sample', 'chunk_len_samples', 'is_padded', 'file_total_samples'],
      dtype='object')


In [28]:
def extract_chunk_embedding(chunk_audio, feature_extractor, model, device="cuda"):

    inputs = feature_extractor(
        chunk_audio.cpu().numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # Stack hidden states: [layers, batch, frames, dim]
    hidden_states = torch.stack(outputs.hidden_states)

    # Softmax over layer weights
    weights = torch.softmax(layer_weights, dim=0)
    weights = weights.view(-1, 1, 1, 1)

    # Weighted sum across layers
    weighted_hidden = (hidden_states * weights).sum(dim=0)

    # Mean pooling over frames (Phase 2: replace with attention)
    embedding = weighted_hidden.mean(dim=1)

    return embedding.squeeze(0).cpu()  # [1024]

In [29]:
import torchaudio
import numpy as np

def extract_embeddings_for_file(file_id):

    audio_path = os.path.join(PROCESSED_AUDIO_PATH, f"{file_id}.wav")
    waveform, sr = torchaudio.load(audio_path)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    waveform = waveform.squeeze(0)

    file_chunks = chunk_df[chunk_df["file_id"] == int(file_id)]

    embeddings = []

    for _, row in file_chunks.iterrows():

        start_sample = int(row["start_sample"])
        end_sample = int(row["end_sample"])

        chunk_audio = waveform[start_sample:end_sample]

        # ---- FIX: ensure 3-second length ----
        expected_len = 48000
        if len(chunk_audio) < expected_len:
            padding = expected_len - len(chunk_audio)
            chunk_audio = torch.nn.functional.pad(chunk_audio, (0, padding))
        # -------------------------------------

        embedding = extract_chunk_embedding(
            chunk_audio,
            feature_extractor,
            model,
            device=device
        )

        embeddings.append(embedding)

    embeddings = torch.stack(embeddings)

    assert embeddings.shape[0] == len(file_chunks)

    return embeddings

In [30]:
test_file = "288"

emb = extract_embeddings_for_file(test_file)

print("Embedding shape:", emb.shape)

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Embedding shape: torch.Size([24, 1024])


In [31]:
print(EMBED_SAVE_PATH)

/content/drive/MyDrive/Hackenza_MUPlovers/cache/embeddings


In [32]:
!ls /content/drive/MyDrive/Hackenza_MUPlovers/cache


embeddings  features_normalised  noise
features    features_normalized  scaler.pkl


In [33]:
import os

EMBED_SAVE_PATH = "/content/drive/MyDrive/Hackenza_MUPlovers/cache/embeddings"

os.makedirs(EMBED_SAVE_PATH, exist_ok=True)

print("Created clean embeddings folder.")

Created clean embeddings folder.


In [34]:
!ls /content/drive/MyDrive/Hackenza_MUPlovers/cache


embeddings  features_normalised  noise
features    features_normalized  scaler.pkl


In [36]:
np.save(
    os.path.join(EMBED_SAVE_PATH, f"{test_file}.npy"),
    emb.detach().cpu().numpy()
)

print("Saved successfully!")

Saved successfully!


In [37]:
!ls /content/drive/MyDrive/Hackenza_MUPlovers/cache/embeddings

288.npy  341.npy  379.npy  414.npy  466.npy  501.npy  552.npy  601.npy	655.npy
294.npy  342.npy  380.npy  416.npy  468.npy  502.npy  560.npy  605.npy	656.npy
298.npy  343.npy  382.npy  419.npy  473.npy  504.npy  565.npy  606.npy	657.npy
299.npy  344.npy  383.npy  420.npy  477.npy  510.npy  566.npy  607.npy	663.npy
309.npy  345.npy  386.npy  430.npy  478.npy  514.npy  568.npy  609.npy	668.npy
311.npy  351.npy  387.npy  431.npy  479.npy  517.npy  569.npy  617.npy	680.npy
316.npy  352.npy  389.npy  433.npy  480.npy  520.npy  571.npy  618.npy	682.npy
317.npy  355.npy  390.npy  435.npy  481.npy  521.npy  573.npy  621.npy	689.npy
319.npy  357.npy  393.npy  440.npy  484.npy  522.npy  574.npy  625.npy	696.npy
322.npy  360.npy  394.npy  441.npy  485.npy  525.npy  576.npy  626.npy	697.npy
326.npy  362.npy  396.npy  446.npy  487.npy  529.npy  578.npy  628.npy	703.npy
328.npy  363.npy  398.npy  451.npy  490.npy  530.npy  579.npy  632.npy	716.npy
331.npy  367.npy  402.npy  452.npy  492.npy  532.npy

In [38]:
import numpy as np
import os

all_files = sorted([
    f.replace(".wav", "")
    for f in os.listdir(PROCESSED_AUDIO_PATH)
    if f.endswith(".wav")
])

print("Total files:", len(all_files))

for file_id in all_files:

    save_path = os.path.join(EMBED_SAVE_PATH, f"{file_id}.npy")

    if os.path.exists(save_path):
        continue

    emb = extract_embeddings_for_file(file_id)

    np.save(save_path, emb.cpu().numpy())

    print(f"Saved {file_id}")

Total files: 160


In [39]:
len(os.listdir(EMBED_SAVE_PATH))

160

In [40]:
test_file = "288"

emb = np.load(os.path.join(EMBED_SAVE_PATH, f"{test_file}.npy"))
prosody = np.load(f"/content/drive/MyDrive/Hackenza_MUPlovers/cache/features/prosody/{test_file}.npy")

print("Embeddings shape:", emb.shape)
print("Prosody shape:", prosody.shape)

Embeddings shape: (24, 1024)
Prosody shape: (24, 10)
